In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from libpysal.weights import Queen
from esda.moran import Moran, Moran_BV, Moran_Local, Moran_Local_BV
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded.")


## Step 1 — Load Data

- `cleaned_bound_spaces.csv` — spatially joined file: each rec facility tagged to its CT
- `census_tract_clipped.zip` — CT polygon boundaries for Queen contiguity weights

In [ ]:
# Joined rec spaces + diabetes data
df = pd.read_csv(
    "https://raw.githubusercontent.com/seven143143/GGR376-Final-Project/main/cleaned_bound_spaces.csv"
)
df = df.dropna(subset=["name", "VALUE3"]).copy()

# CT boundary shapefile (upload to same directory or adjust path)
gdf = gpd.read_file("census_tract_clipped.zip")

print(f"Rec space rows : {len(df):,}")
print(f"CT polygons    : {len(gdf):,}")
print(f"Shapefile CRS  : {gdf.crs}")


## Step 2 — Aggregate to Census Tract Level

Group rec space rows by `spatial_id` (CT UID) to get facility count and diabetes % per tract.

In [ ]:
tract_summary = (
    df.groupby("spatial_id")
    .agg(
        num_spaces   = ("OBJECTID", "count"),
        diabetes_pct = ("VALUE3",   "first"),
    )
    .reset_index()
)

print(f"Census tracts in summary: {len(tract_summary)}")
tract_summary.describe()


## Step 3 — Join Tract Summary to CT Polygons

Merge the tract-level data onto the shapefile using `CTUID` ↔ `spatial_id`. Reproject to **UTM Zone 17N (EPSG:32617)** for accurate distance calculations.

In [ ]:
# Standardise CTUID to float for matching
gdf["CTUID_float"] = gdf["CTUID"].astype(float)

# Inner join — keeps only tracts present in both datasets (952 tracts)
gdf_joined = gdf.merge(
    tract_summary,
    left_on  = "CTUID_float",
    right_on = "spatial_id",
    how      = "inner"
).to_crs("EPSG:32617")

print(f"Matched census tracts: {len(gdf_joined)}")
gdf_joined[["CTUID", "num_spaces", "diabetes_pct"]].head()


## Step 4 — Pearson Correlation & Scatter Plot

In [ ]:
x = gdf_joined["num_spaces"]
y = gdf_joined["diabetes_pct"]

corr, p_corr = stats.pearsonr(x, y)
print(f"Pearson r = {corr:.4f}  |  p = {p_corr:.4f}")

m, b    = np.polyfit(x, y, 1)
x_line  = np.linspace(x.min(), x.max(), 200)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x, y, color="#4C8BB5", alpha=0.45, edgecolors="white",
           linewidths=0.3, s=30, label="Census tracts")
ax.plot(x_line, m * x_line + b, color="#E05C3A", linewidth=2,
        label=f"Best-fit (slope = {m:.4f})")

textstr = f"Pearson r = {corr:.3f}\np = {p_corr:.4f}\nn = {len(gdf_joined)} tracts"
props   = dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, edgecolor="#ccc")
ax.text(0.97, 0.95, textstr, transform=ax.transAxes, fontsize=10,
        va="top", ha="right", bbox=props)

ax.set_xlabel("Number of recreational spaces in census tract", fontsize=12)
ax.set_ylabel("Diabetes percentage (%)", fontsize=12)
ax.set_title("Recreational Spaces vs. Diabetes Percentage\nby Census Tract",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## Step 5 — Queen Contiguity Spatial Weights

Two census tracts are neighbours if they **share any border or corner** (Queen contiguity). This uses the actual CT polygon geometry — the methodologically correct approach for areal data.

Weights are **row-standardised** so each row sums to 1.

In [ ]:
W = Queen.from_dataframe(gdf_joined)
W.transform = "r"

print(f"Census tracts : {W.n}")
print(f"Non-zero links: {W.nonzero}")
print(f"Islands (0 neighbours): {len(W.islands)}")
print(f"Mean neighbours : {np.mean([len(v) for v in W.neighbors.values()]):.2f}")
print(f"Min  neighbours : {W.min_neighbors}")
print(f"Max  neighbours : {W.max_neighbors}")


## Step 6 — Standardise Variables

In [ ]:
X_raw = gdf_joined["diabetes_pct"].values.astype(float)
Y_raw = gdf_joined["num_spaces"].values.astype(float)

X_z = (X_raw - X_raw.mean()) / X_raw.std(ddof=1)
Y_z = (Y_raw - Y_raw.mean()) / Y_raw.std(ddof=1)

print(f"Diabetes (X)   — mean: {X_raw.mean():.3f}%,  std: {X_raw.std(ddof=1):.3f}%")
print(f"Rec Spaces (Y) — mean: {Y_raw.mean():.1f},    std: {Y_raw.std(ddof=1):.1f}")


## Step 7 — Univariate Moran's I

Is each variable spatially clustered on its own?
- **I > 0** → similar values cluster together
- **I ≈ 0** → spatially random
- **I < 0** → dissimilar values cluster

In [ ]:
np.random.seed(42)
PERMS = 999

mi_x = Moran(X_z, W, permutations=PERMS)
mi_y = Moran(Y_z, W, permutations=PERMS)

print("=" * 60)
print("UNIVARIATE MORAN'S I  (Queen contiguity, 999 permutations)")
print("=" * 60)
print(f"{'Variable':<28} {'I':>7}  {'E[I]':>7}  {'p-value':>8}  {'Sig?':>5}")
print("-" * 60)
print(f"{'Diabetes %':<28} {mi_x.I:>7.4f}  {mi_x.EI:>7.4f}  {mi_x.p_sim:>8.4f}  {'Yes' if mi_x.p_sim < 0.05 else 'No':>5}")
print(f"{'Recreational Spaces':<28} {mi_y.I:>7.4f}  {mi_y.EI:>7.4f}  {mi_y.p_sim:>8.4f}  {'Yes' if mi_y.p_sim < 0.05 else 'No':>5}")


## Step 8 — Bivariate Moran's I

Measures the spatial correlation between **diabetes at location *i*** and the **spatial lag of recreational spaces at neighbouring tracts *j***.

$$I_{BV} = \frac{\sum_i z_{x,i} \sum_j w_{ij} z_{y,j}}{n}$$

In [ ]:
np.random.seed(42)
bv = Moran_BV(X_z, Y_z, W, permutations=PERMS)

print("=" * 60)
print("BIVARIATE MORAN'S I  (X = Diabetes, Y = Rec Spaces)")
print("=" * 60)
print(f"  Moran's I  = {bv.I:.4f}")
print(f"  z-score    = {bv.z_sim:.4f}")
print(f"  p-value    = {bv.p_sim:.4f}  ({PERMS} permutations)")
print(f"  Significant: {'Yes' if bv.p_sim < 0.05 else 'No'}  (alpha = 0.05)")

if bv.p_sim < 0.05:
    if bv.I < 0:
        print()
        print("  Interpretation: Significant NEGATIVE spatial autocorrelation.")
        print("  High-diabetes tracts tend to be surrounded by tracts with")
        print("  FEWER recreational spaces — a pattern of spatial inequity.")


## Step 9 — LISA (Local Indicators of Spatial Association)

LISA decomposes the global Moran's I into tract-level scores, showing **where** clusters are located. Each tract is assigned a quadrant (only shown if p < 0.05):

| Quadrant | Label | Meaning |
|---|---|---|
| 1 | **HH** | High value, high neighbours — hotspot |
| 2 | **LH** | Low value, high neighbours — spatial outlier |
| 3 | **LL** | Low value, low neighbours — coldspot |
| 4 | **HL** | High value, low neighbours — spatial outlier |

In [ ]:
np.random.seed(42)

lisa_x  = Moran_Local(X_z, W, permutations=PERMS)
lisa_bv = Moran_Local_BV(X_z, Y_z, W, permutations=PERMS)

sig_x  = lisa_x.p_sim  < 0.05
sig_bv = lisa_bv.p_sim < 0.05

quad_labels = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}

print(f"Univariate LISA (Diabetes) — significant tracts : {sig_x.sum()}")
for q, label in quad_labels.items():
    print(f"  {label}: {((lisa_x.q == q) & sig_x).sum()}")

print()
print(f"Bivariate LISA (Diabetes → Rec Spaces) — significant: {sig_bv.sum()}")
for q, label in quad_labels.items():
    print(f"  {label}: {((lisa_bv.q == q) & sig_bv).sum()}")


## Step 10 — Visualisations

In [ ]:
quad_colors = {1: "#D73027", 2: "#91BFDB", 3: "#4575B4", 4: "#FC8D59"}
ns_color    = "#d3d3d3"

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Spatial Analysis — Diabetes vs Recreational Spaces\n(Queen Contiguity Weights)",
             fontsize=14, fontweight="bold", y=1.01)

# ── Plot 1: Bivariate Moran scatter ──────────────────────────────────────────
ax = axes[0, 0]
Wy = W.sparse.dot(Y_z)

qc = []
for xi, wyi in zip(X_z, Wy):
    if   xi >= 0 and wyi >= 0: qc.append("#D73027")
    elif xi <  0 and wyi <  0: qc.append("#4575B4")
    elif xi >= 0 and wyi <  0: qc.append("#FC8D59")
    else:                       qc.append("#91BFDB")

ax.scatter(X_z, Wy, c=qc, alpha=0.45, edgecolors="white", linewidths=0.2, s=12)
xl = np.linspace(X_z.min(), X_z.max(), 200)
sl, ic = np.polyfit(X_z, Wy, 1)
ax.plot(xl, sl*xl+ic, "k--", lw=1.5, label=f"slope = {sl:.3f}")
ax.axhline(0, color="grey", lw=0.7); ax.axvline(0, color="grey", lw=0.7)
patches = [mpatches.Patch(color=c, label=l) for c, l in
           zip(["#D73027","#4575B4","#FC8D59","#91BFDB"], ["HH","LL","HL","LH"])]
ax.legend(handles=patches, fontsize=8)
ax.set_xlabel("Diabetes % (z-score)"); ax.set_ylabel("Spatial Lag — Rec Spaces (z-score)")
ax.set_title(f"Bivariate Moran Scatter  (I = {bv.I:.4f}, p = {bv.p_sim:.4f})", fontsize=10)

# ── Plot 2: Permutation distribution ─────────────────────────────────────────
ax = axes[0, 1]
ax.hist(bv.sim, bins=50, color="#A8C6E0", edgecolor="white", density=True)
ax.axvline(bv.I, color="#D73027", lw=2.5, label=f"Observed I = {bv.I:.4f}")
ax.axvline(np.mean(bv.sim), color="#555", lw=1.5, ls="--",
           label=f"Mean sim = {np.mean(bv.sim):.4f}")
ax.set_xlabel("Simulated Moran's I"); ax.set_ylabel("Density")
ax.set_title("Permutation Reference Distribution (999 perms)", fontsize=10)
ax.legend(fontsize=9)

# ── Plot 3: Univariate LISA map ───────────────────────────────────────────────
ax = axes[1, 0]
gdf_joined["lisa_x_q"]   = lisa_x.q
gdf_joined["lisa_x_sig"] = sig_x

for q, color in quad_colors.items():
    mask = (gdf_joined["lisa_x_q"] == q) & gdf_joined["lisa_x_sig"]
    if mask.sum() > 0:
        gdf_joined[mask].plot(ax=ax, color=color, linewidth=0.2,
                              edgecolor="white", label=f"{quad_labels[q]} (n={mask.sum()})")

gdf_joined[~gdf_joined["lisa_x_sig"]].plot(ax=ax, color=ns_color,
                                            linewidth=0.2, edgecolor="white",
                                            label="Not significant")
ax.set_title("Univariate LISA — Diabetes %\n(p < 0.05)", fontsize=10, fontweight="bold")
ax.set_axis_off()
ax.legend(fontsize=7.5, loc="lower right", framealpha=0.9)

# ── Plot 4: Bivariate LISA map ────────────────────────────────────────────────
ax = axes[1, 1]
bv_labels = {1: "HH", 2: "LH", 3: "LL", 4: "HL ⚠"}
gdf_joined["lisa_bv_q"]   = lisa_bv.q
gdf_joined["lisa_bv_sig"] = sig_bv

for q, color in quad_colors.items():
    mask = (gdf_joined["lisa_bv_q"] == q) & gdf_joined["lisa_bv_sig"]
    if mask.sum() > 0:
        gdf_joined[mask].plot(ax=ax, color=color, linewidth=0.2,
                              edgecolor="white", label=f"{bv_labels[q]} (n={mask.sum()})")

gdf_joined[~gdf_joined["lisa_bv_sig"]].plot(ax=ax, color=ns_color,
                                             linewidth=0.2, edgecolor="white",
                                             label="Not significant")
ax.set_title("Bivariate LISA — Diabetes (X) → Rec Spaces (Y)\n(p < 0.05)",
             fontsize=10, fontweight="bold")
ax.set_axis_off()
ax.legend(fontsize=7.5, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()


## Summary

In [ ]:
summary = pd.DataFrame({
    "Test": [
        "Pearson r (aspatial)",
        "Univariate Moran's I — Diabetes",
        "Univariate Moran's I — Rec Spaces",
        "Bivariate Moran's I (X=Diabetes, Y=Rec Spaces)",
    ],
    "Statistic": [
        f"r = {corr:.4f}", f"I = {mi_x.I:.4f}",
        f"I = {mi_y.I:.4f}", f"I = {bv.I:.4f}"
    ],
    "p-value": [
        f"{p_corr:.4f}", f"{mi_x.p_sim:.4f}",
        f"{mi_y.p_sim:.4f}", f"{bv.p_sim:.4f}"
    ],
    "Sig. (a=0.05)": [
        "Yes" if p_corr     < 0.05 else "No",
        "Yes" if mi_x.p_sim < 0.05 else "No",
        "Yes" if mi_y.p_sim < 0.05 else "No",
        "Yes" if bv.p_sim   < 0.05 else "No",
    ]
})
summary
